# PyTorch Attention

PyTorch provides ready-made attention algorithms, or you can build from scratch for full control.

---

# 1. Ready-Made Algorithms

## 1.1 MultiheadAttention (MHA)

**What it does:**
- Creates Q, K, V projections (linear layers)
- Splits into multiple heads
- Computes attention per head
- Concatenates and projects back

```python
import torch
import torch.nn as nn

mha = nn.MultiheadAttention(
    embed_dim=768,          # d_model (required)
    num_heads=12,           # split d_model into N heads (required)
    dropout=0.1,            # regularization
    bias=True,              # linear layers use bias
    batch_first=True        # (B, T, D) format, not (T, B, D)
)

x = torch.randn(32, 10, 768)  # (batch, seq, d_model)
output, attention_weights = mha(x, x, x)  # Q, K, V
```

**For decoder (causal mask):**
```python
mask = torch.triu(torch.ones(10, 10), diagonal=1).bool()
output, weights = mha(x, x, x, attn_mask=mask)
```

---

## 1.2 Scaled Dot-Product Attention (SDPA)

**What it does:**
- Computes attention scores: QK^T / √d_k
- Applies mask (optional)
- Applies softmax
- Multiplies by values

```python
import torch.nn.functional as F

Q = torch.randn(32, 12, 10, 64)  # (batch, heads, seq, d_k)
K = torch.randn(32, 12, 10, 64)
V = torch.randn(32, 12, 10, 64)

# Forward
output = F.scaled_dot_product_attention(
    query=Q,
    key=K,
    value=V,
    attn_mask=None,         # custom mask
    dropout_p=0.1,          # dropout during training
    is_causal=False         # True for decoder (auto causal mask)
)
# output: (32, 12, 10, 64) - same as input
```

**Causal (decoder only):**
```python
output = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
```

---

## 1.3 Converting SDPA output back to (B, T, D)

```python

# Transpose and reshape
output = output.transpose(1, 2)        # (batch, seq, heads, d_k)
output = output.reshape(batch, seq, d_model)  # (batch, seq, d_model)
```

---

## 1.4 Communication between heads (output projection)

```python
import torch.nn as nn

d_model = 768
output_projection = nn.Linear(d_model, d_model)

# After SDPA, project back
output = output.reshape(batch, seq, d_model)
output = output_projection(output)  # (batch, seq, d_model)
```

---

# 2. Hands-On: Building Attention from Scratch

## 2.1 Single-Head Attention

```python
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class SimpleAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        
        # Linear layers for Q, K, V
        self.W_q = nn.Linear(d_model, d_model, bias = False)
        self.W_k = nn.Linear(d_model, d_model, bias = False)
        self.W_v = nn.Linear(d_model, d_model, bias = False)
        self.W_o = nn.Linear(d_model, d_model, bias = False)
    
    def forward(self, x, mask=None):
        # x: (batch, seq, d_model)
        
        # Project to Q, K, V
        Q = self.W_q(x)  # (batch, seq, d_model)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # Attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_model) # SHAPE: (batch, seq, seq)

        # Mask if decoder
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Softmax
        scores = F.softmax(scores, dim=-1)
        
        # Multiply by values
        output = scores @ V  # (batch, seq, d_model)
        
        # Output projection
        output = self.W_o(output)
        
        return output
```


---

## 2.2 MultiHead Attention from Scratch

```python
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads # D_K It's the d_model per head 
        
        self.W_q = nn.Linear(d_model, d_model, bias = False)
        self.W_k = nn.Linear(d_model, d_model, bias = False)
        self.W_v = nn.Linear(d_model, d_model, bias = False)
        self.W_o = nn.Linear(d_model, d_model, bias = False)
    
    
    def forward(self, x, mask=None):
        b, t, c = x.shape
        # b = batches
        # t = tokens
        # c = d_model
        
        # Project
        Q = self.split_heads(self.W_q(x))  # (batch, heads, t, d_k)
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        # Split heads
        Q = Q.reshape(b, t, self.num_heads, self.d_k).transpose(1, 2)
        K = K.reshape(b, t, self.num_heads, self.d_k).transpose(1, 2)
        V = V.reshape(b, t, self.num_heads, self.d_k).transpose(1, 2)

        
        # Attention
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Mask if decoder
        if mask is not None:
            mask = torch.tril(torch.ones(t, t, device = x.device ))
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Softmax
        scores = F.softmax(scores, dim=-1)
        
        # Apply to values
        out = scores @ V  # (batch, heads, seq, d_k)
        
        # Concat heads
        out = out.transpose(1, 2).reshape(batch, -1, self.d_model)  
        # Transpose (batch, seq, heads, d_k)
        # Reshape (batch, seq, d_model)
        
        # Output
        out = self.W_o(out)
        
        return out
```

---

## 2.3 When to use what?

| Algorithm | When | Why |
|-----------|------|-----|
| **MHA** | Production, simple case | Easy, fast, optimized |
| **SDPA** | Custom control, performance critical | Flexible, efficient |
| **From scratch** | Learning, debugging, custom logic | Full understanding |

---

# Summary

1. **MHA**: Black box, (B,T,D) in → (B,T,D) out
2. **SDPA**: More control, handle heads manually
3. **Scratch**: Full control, learning tool

Start with MHA, move to SDPA if needed, build from scratch to understand.

---

# Now Let's analyze better this..

# 1. Create the QKV Matriz

We gonna use the PyTorch `Linear()` for get the QKV matrix, easy and fast, the args are

```python

wX = nn.linear(
    in_features = 1, # The values of features 
    out_features = 10, # The value of neurons (In this context how many QUESTIONS/KEYS/VALUES gonna have)
    bias = False, # Normally in attention we don't use bias
)

```

In [ ]:
# Creation the QKV Matrix
wQ = nn.Linear(d_model, d_model)
wK = nn.Linear(d_model, d_model)
wV = nn.Linear(d_model, d_model)

# Get the QKV
Q = wQ(x)
K = wK(x)
V = wV(x)


---
# 2. Get the RAW Scores

We gonna use the PyTorch `@` for get the RAW SCORE matrix, easy and fast

```python

raw_scores = (Q @ K.tranpose(-1, -2)) / math.sqrt(d_k)


```

In [ ]:
# Get the raw scores
raw_scores = (Q @ K.tranpose(-1, -2)) / math.sqrt(d_k)

---
# 3.  Applies the mask

### 1. Why the mask must have shape `(T, T)`

After the raw_scores the matrix gonna have the shape (B, T, T)

```
Q @ K.T  →  [T, d_k] @ [d_k, T]  →  [T, T]
```

Since the mask is going to be applied directly on top of `scores`, **it must have exactly the same shape**: `(T, T)`. This makes sense because the mask is a cell-by-cell map of "who is allowed to look at whom," and that map only exists in the query-key relationship — i.e., in the scores matrix itself.

So basically we need to number of tokens that have in the sequence

```python

T = scores.shape[-1]  # sequence length or
b, t, c = x.shape

mask = torch.tril(torch.ones(T, T))
```

### 2. Tril vs Triu

Both are functions that **zero out one half of a matrix**, splitting it along the main diagonal. The difference is which half each one keeps.

1. **TRIL** (lower triangular)

Keeps the main diagonal **and everything below it**. Zeroes out everything above it.

```python

torch.tril(torch.ones(4, 4))

"""

1 0 0 0
1 1 0 0
1 1 1 0
1 1 1 1

"""

```

Interpretation in attention: row`i (the query at position i) only keeps a value of 1 in columns j <= i. In other words, **a query can only see keys at the same position or earlier** — exactly the causal behavior we want (no peeking into the future).

2. **TRIU**  (upper triangular)

Keeps the main diagonal **and everything above it**. Zeroes out everything below it.

```python

torch.triu(torch.ones(4, 4))

"""

1 1 1 1
0 1 1 1
0 0 1 1
0 0 0 1

"""

```

This is the opposite: each position would only see the future, never the past — not what we use for causal masking, but useful, for instance, to generate the "inverted" version of a mask or in other algorithms that need the upper half of a matrix.


### 3. Applying the mask with `masked_fill`

`masked_fill(mask, value)` works like an "if/else" applied cell by cell:

where mask[i][j] == True  → replace with value
where mask[i][j] == False → keep the original value


The catch: our causal_mask has True in the **allowed** positions (past/present), but `masked_fill` replaces exactly where it receives `True`. If we applied it directly, we'd be blocking the past and leaving the future open — the opposite of what we want.

That's why we invert the mask before applying it:

```python

causal_mask = torch.tril(torch.ones(T, T, dtype=torch.bool))
scores = scores.masked_fill(~causal_mask, float('-inf')) # masked_fill applies -inf where it receives True → lands exactly on what was blocked

```


### 4. Why `-inf` and not `0`

After masking, the next step is **softmax**, applied row by row:

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

If we filled the blocked positions with `0` instead of `-inf`, the token would still receive a **non-zero** probability after softmax, because:

$$e^0 = 1$$

In other words, the blocked position would still contribute (with a small but nonzero weight) to the final result — a leak of future information, even if small.

With `-inf`:

$$e^{-\infty} = 0$$

The blocked position receives **exactly zero** weight after softmax. It stops contributing at all to the final attn_weights @ V. This is the only way to guarantee a *complete* block, not just an "attenuated" one.

### Comparing the two cases

```python
scores_with_zero = torch.tensor([2.0, 0.0])       # "weak" block
scores_with_inf  = torch.tensor([2.0, float('-inf')])  # real block

torch.softmax(scores_with_zero, dim=0)
# tensor([0.8808, 0.1192])   ← still leaks ~12% of attention

torch.softmax(scores_with_inf, dim=0)
# tensor([1.0000, 0.0000])   ← absolute zero, no leakage
```


In [ ]:
ones = torch.ones(4, 4, dtype=bool) # The matrix must have the shape (T, T)
mask = torch.tril(ones)
scores = torch.randn((4, 4))
scores = scores.masked_fill(mask == False , float('-inf')) # Or we can use the ~mask
print(ones)
print(mask)
print(scores)

tensor([[True, True, True, True],
        [True, True, True, True],
        [True, True, True, True],
        [True, True, True, True]])
tensor([[ True, False, False, False],
        [ True,  True, False, False],
        [ True,  True,  True, False],
        [ True,  True,  True,  True]])
tensor([[ 0.4833,    -inf,    -inf,    -inf],
        [-1.0084, -0.0633,    -inf,    -inf],
        [-0.2471,  0.5808,  0.3708,    -inf],
        [-2.1342,  0.8024,  1.1455, -0.1357]])



---
# 4. Softmax - Quick Rule

`dim` = **axis that collapses**, not the one you keep.

```python
scores.sum(dim=-1)  # Sums columns → result per ROW
scores.sum(dim=0)   # Sums rows → result per COLUMN
```


## Quick Rule

| You want... | Use... | Why |
|---|---|---|
| Each **row** sums to 1 | `dim=-1` (columns) | Values in row span columns |
| Each **column** sums to 1 | `dim=0` (rows) | Values in column span rows |


## In Attention

```python
scores.shape  # [batch, heads, seq_q, seq_k]

attn = torch.softmax(scores, dim=-1)
# Each query (row) normalizes across keys (columns)
```

**`dim=-1`** always = normalize over **last axis**.

**TL;DR:** `dim` = axis that disappears, not the one that stays. In attention: `softmax(scores, dim=-1)`.

In [ ]:
scores_softmax = F.softmax(scores, dim=-1)
print(scores_softmax)
print(scores.shape)

tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.2799, 0.7201, 0.0000, 0.0000],
        [0.1944, 0.4449, 0.3606, 0.0000],
        [0.0186, 0.3504, 0.4939, 0.1371]])
torch.Size([4, 4])


---
# 5. Values and Scores (@)

After softmax, scores are probabilities (each row sums to 1).

```python

scores = torch.softmax(scores, dim=-1) # (batch, heads, seq_q, seq_k)

output = scores @ V # (batch, heads, seq_q, d_k)
```

In [ ]:
output = scores_softmax @ V

Scores result:
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.2799, 0.7201, 0.0000, 0.0000],
        [0.1944, 0.4449, 0.3606, 0.0000],
        [0.0186, 0.3504, 0.4939, 0.1371]])
Output result:
tensor([[0.8066, 0.6574, 0.7097, 0.6223],
        [0.9038, 0.4097, 0.6067, 0.1939],
        [0.6189, 0.2970, 0.6797, 0.2724],
        [0.4671, 0.2558, 0.6549, 0.3095]])


---
# 6. Concat Heads (If needed)

After attention, you have multiple heads:

```python
output.shape  # (batch, heads, seq, d_k)
```

To get back to (batch, seq, d_model):

```python
batch, heads, seq, d_k = output.shape
d_model = heads * d_k

# Transpose: (batch, heads, seq, d_k) → (batch, seq, heads, d_k)
output = output.transpose(1, 2)

# Reshape: (batch, seq, heads, d_k) → (batch, seq, d_model)
output = output.reshape(batch, seq, d_model)
```


## Then project (output layer)

```python
W_o = nn.Linear(d_model, d_model)
output = W_o(output)
# (batch, seq, d_model)
```

This allows heads to communicate (combine information).

---

# 7. Output Projection (If needed)

After concatenating heads, project back to d_model:

```python
output.shape  # (batch, seq, d_model)

W_o = nn.Linear(d_model, d_model)
output = W_o(output)
# (batch, seq, d_model)
```

**Why?**
- Allows heads to **communicate** (mix information)
- Learns which head outputs are important
- Required for multi-head attention to work
